# Assignment 20 - Movie Recommendation System

Content based recommender using TF-IDF and cosine similarity.

Dataset: TMDB 5000 Movie Dataset (`tmdb_5000_movies.csv`)
Kaggle link: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

## Part 1 - Data Preprocessing

### Task 1: Load and understand the dataset

In [ ]:
import json
import re

import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_csv("tmdb_5000_movies.csv")
print("Shape:", df.shape)

In [ ]:
print(df.columns.tolist())

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
print("duplicate titles:", df["title"].duplicated().sum())
df[["budget", "popularity", "runtime", "vote_average", "vote_count"]].describe()

In [ ]:
# what the text columns look like for one movie
print(df.loc[0, "title"])
print(df.loc[0, "overview"])
print(df.loc[0, "genres"])
print(df.loc[0, "keywords"][:200])

The item identifier is `title`.

For recommendations I am using these text columns:
- `overview` - the plot summary, this has most of the information
- `tagline` - one line about the movie
- `genres` - list of genres (stored as a JSON string)
- `keywords` - list of keywords (also JSON)

`homepage` has a lot of missing values but I don't need it. `overview` has 3 missing and `tagline` has around 800 missing, these are replaced with an empty string in the next step.

### Task 2: Text preprocessing

In [ ]:
movies = df[["title", "overview", "genres", "keywords", "tagline"]].copy()

# a few titles are repeated, keeping the first one so that title lookup is unique
movies = movies.drop_duplicates(subset="title").reset_index(drop=True)

# missing values -> empty string
movies = movies.fillna("")
movies.isnull().sum()

In [ ]:
def get_names(text):
    # '[{"id": 28, "name": "Action"}, {"id": 878, "name": "Science Fiction"}]' -> 'Action ScienceFiction'
    # spaces inside a name are removed so "Science Fiction" is treated as one word
    try:
        items = json.loads(text)
    except (TypeError, ValueError):
        return ""
    return " ".join(i["name"].replace(" ", "") for i in items)

movies["genres"] = movies["genres"].apply(get_names)
movies["keywords"] = movies["keywords"].apply(get_names)
movies[["title", "genres", "keywords"]].head()

In [ ]:
# combine everything into one text column
movies["tags"] = movies["overview"] + " " + movies["tagline"] + " " + movies["genres"] + " " + movies["keywords"]
movies["tags"][0]

In [ ]:
def clean(text):
    text = str(text).lower()                      # lowercase
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # remove punctuation and special characters
    words = [w for w in text.split() if w not in ENGLISH_STOP_WORDS]   # remove stopwords
    return " ".join(words)

movies["clean_text"] = movies["tags"].apply(clean)
movies["clean_text"][0]

In [ ]:
movies[["title", "clean_text"]].head()

Stopwords are taken from scikit-learn (`ENGLISH_STOP_WORDS`) since only pandas and scikit-learn are allowed, so no nltk.

## Part 2 - Text Vectorization

### Task 3: TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(movies["clean_text"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

In [ ]:
print(tfidf.get_feature_names_out()[1000:1030])

- `max_features=5000` keeps the 5000 most frequent terms. Without a limit the vocabulary with bigrams goes into lakhs and most of those terms appear in only one movie.
- `ngram_range=(1, 2)` uses single words and pairs of words, so things like "new york" or "serial killer" are also captured.

### Task 4: Cosine similarity

In [ ]:
similarity = cosine_similarity(tfidf_matrix)
print(similarity.shape)
similarity[:5, :5].round(3)

**Why cosine similarity?**

Cosine similarity measures the angle between two vectors and not their length. Some movies have a long overview and some have only one line, so their TF-IDF vectors have very different magnitudes. With euclidean distance the long ones would look far from everything. With cosine only the direction matters, i.e. which words are important in both movies. The value is between 0 and 1 here (TF-IDF has no negative values), 1 means same content and 0 means nothing in common, so it is easy to sort on. It also works fast on sparse matrices.

## Part 3 - Recommendation Logic

### Task 5: Recommendation function

In [ ]:
def recommend(item_name, top_n=5):
    if item_name not in movies["title"].values:
        print(item_name, "is not in the dataset")
        return []

    # index of the selected movie
    idx = movies[movies["title"] == item_name].index[0]

    # similarity of this movie with all the others
    scores = list(enumerate(similarity[idx]))

    # sort by score, first one is the movie itself so skip it
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n + 1]

    return [(movies["title"][i], round(float(s), 3)) for i, s in scores]

In [ ]:
recommend("Avatar")

In [ ]:
recommend("The Dark Knight Rises")

In [ ]:
recommend("Toy Story")

In [ ]:
recommend("Harry Potter and the Half-Blood Prince", top_n=8)

In [ ]:
recommend("Some random movie")

The results make sense, sequels and movies from the same franchise or with the same theme come up at the top.

The Streamlit app is in `app.py` and uses the same steps.